# Hybrid Quantum CNN — Multi-seed statistical campaign (v1)

Notebook QCNN per il confronto inferenziale del Cap.3 (Wave K) della tesi.
Architettura: modello Hybrid Q-CNN `C16-Q64` di Filippi (tesi magistrale, Pisa AA 2024/2025, supervisori Morsch + Cappuccio).

- **Circuito**: 9 qubit, RY encoding → H → CNOT staircase → RZ trainable → inv CNOT 
- **Rete**: Conv(3→16) → Pool → Conv(16→32) → Pool → Conv(32→64) → Quanv(64→64) → FC head
- **Dropout**: 5% (Filippi C16-Q64)
- **Dataset**: EuroSAT, 2 classi (coerente con Filippi §6.3.1), 40 epoche
- **Misura quantum**: singolo qubit (qubit 0)
- **Multi-seed**: $R=10$ run con seed `42 + run_idx * 111`
- **Output**: `Output_QCNN_v1_multiseed/results.json` (formato esteso con predizioni per-item)

Il confronto Wilcoxon paired QCNN-vs-CCNN (CCNN matched-capacity, vedi `classical_cnn_multiseed_stats_v1.ipynb`) viene eseguito nella §18.7.

## §1 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time, os, csv, copy, random, gc
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, Literal, List, Dict
from pathlib import Path

import pytorch_lightning as L
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import Callback, ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchmetrics import Accuracy
from PIL import Image

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorEstimator

# AerSimulator — fallback se non installato
try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.primitives import EstimatorV2 as AerEstimator
    HAS_AER = True
except ImportError:
    HAS_AER = False
    print("⚠️  qiskit-aer non trovato — solo StatevectorEstimator disponibile")

QISKIT_VERSION = tuple(int(x) for x in qiskit.__version__.split('.')[:2])
assert QISKIT_VERSION >= (2, 0), f"Richiesto Qiskit >= 2.0, trovato {qiskit.__version__}"
print(f"Qiskit {qiskit.__version__} | PyTorch {torch.__version__} | Lightning {L.__version__}")
print(f"AerSimulator: {'✓' if HAS_AER else '✗'}")

## §2 — Configurazione 

Architettura: Conv1(3→6,k5) → Pool → Conv2(6→6,k5) → Pool → Quanv(9q, 3×3) → Flatten → FC

In [ ]:
@dataclass
class QCNNConfig:
    """Configurazione — riproduzione Filippi."""

    # ── Backend ──
    backend_type: Literal["statevector", "aer", "aer_noise", "ibm"] = "statevector"  # StatevectorEstimator: piu\' veloce di Aer per 9 qubit ideali (ottimizzazione Wave K)
    ibm_token: Optional[str] = None
    ibm_backend_name: Optional[str] = None
    ibm_instance: Optional[str] = None
    ibm_channel: str = "ibm_quantum"
    ibm_min_qubits: int = 127
    optimization_level: int = 1
    noise_backend_name: str = "ibm_brisbane"

    # ── Circuito quantistico  ──
    num_qubits: int = 9                       # 3×3 kernel → 9 qubit
    kernel_size: int = 3                      # 3×3 quantum kernel
    stride: int = 1                           # stride convolution quantistica
    quanv_padding: int = 0                    # 0 = no padding (16→14×14)                    # padding per quanv (16→18)
    measure_qubit: int = 0                    # qubit da misurare (: singolo qubit)
    shots: int = 0                            # 0 = esatto (statevector)

    # ── Rete classica  ──
    num_conv_channels: int = 6                # canali fissi a 6 
    conv_kernel_size: int = 5                 # kernel classico 5×5
    conv_padding: int = 2                     # padding per mantenere dimensioni
    dropout_rate: float = 0.0                 # 0 senza dropout, 0.05 con dropout (matched a Filippi C16-Q64)

    # ── Dataset EuroSAT ──
    train_dir: str = "./dataset/training"
    val_dir: str = "./dataset/validation"
    img_size: int = 64
    in_channels: int = 3
    num_classes: int = 2                      # 2 classi (Forest vs AnnualCrop), coerente con Filippi §6.3.1
    selected_classes: Optional[List[str]] = None  # None = prime 4 disponibili
    max_samples_per_class: Optional[int] = 100   # 100 per classe (coerente con run effettivo CCNN, N_val=200)

    # ── Training ──
    batch_size: int = 16
    max_epochs: int = 10                      # 10 epoche (coerente con run effettivo, vedi sec:stats-significance Cap.3)
    lr: float = 0.001
    weight_decay: float = 1e-4
    num_workers: int = 4
    early_stop_patience: int = 12              # Disabilitato di fatto per 10 epoche             # Filippi non usa early stopping

    # ── Loop statistico ──
    num_stat_runs: int = 10                   # R=10 run multi-seed (Cap.3 Wave K)
    base_seed: int = 42

    # ── I/O ──
    run_name: str = "filippi_v2"
    output_dir: str = "Output_QCNN_v1_multiseed"
    seed: int = 42

    # ── Proprietà calcolate ──
    @property
    def num_weights(self) -> int:
        """Pesi trainabili circuito Filippi: 9 RZ gates."""
        return self.num_qubits

    @property
    def feature_map_size(self) -> int:
        """Dimensione feature map dopo 2 pool: 64 → 32 → 16."""
        return self.img_size // 4

    @property
    def quanv_output_size(self) -> int:
        """Output spatial size dopo quantum conv."""
        fm = self.feature_map_size
        return (fm + 2 * self.quanv_padding - self.kernel_size) // self.stride + 1

    @property
    def flatten_size(self) -> int:
        return self.num_conv_channels * self.quanv_output_size ** 2

    @property
    def patches_per_channel(self) -> int:
        return self.quanv_output_size ** 2


# ═══════════════════════════════════════════
config = QCNNConfig()
L.seed_everything(config.seed)

print(f"Architettura Filippi:")
print(f"  Circuito: {config.num_qubits}q (kernel {config.kernel_size}×{config.kernel_size})")
print(f"  Pesi quantistici: {config.num_weights} (RZ gates)")
print(f"  Feature map pre-quanv: {config.feature_map_size}×{config.feature_map_size}")
print(f"  Quanv output: {config.quanv_output_size}×{config.quanv_output_size}")
print(f"  Flatten: {config.flatten_size} = {config.num_conv_channels}×{config.quanv_output_size}²")
print(f"  Classi: {config.num_classes}")

## §3 — EuroSAT Dataset (4 classi, full)

In [ ]:
class EuroSATDataset(Dataset):
    """EuroSAT con supporto selezione classi + limite campioni."""

    def __init__(self, root_dir, transform=None, num_classes=4,
                 selected_classes=None, max_samples_per_class=None, seed=42):
        self.root_dir = root_dir
        self.transform = transform
        self.rng = random.Random(seed)

        available = sorted([d for d in os.listdir(root_dir)
                           if os.path.isdir(os.path.join(root_dir, d))])

        if selected_classes:
            self.classes = [c for c in selected_classes if c in available]
        else:
            self.classes = available[:num_classes]

        self.data = []
        for cls_idx, cls_name in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls_name)
            imgs = sorted([f for f in os.listdir(cls_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))])
            if max_samples_per_class and len(imgs) > max_samples_per_class:
                self.rng.shuffle(imgs)
                imgs = imgs[:max_samples_per_class]
            for img_file in imgs:
                self.data.append((os.path.join(cls_path, img_file), cls_idx))

        self.rng.shuffle(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


class EuroSATDataModule(L.LightningDataModule):
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, config: QCNNConfig):
        super().__init__()
        self.config = config
        self.class_names = None

    def setup(self, stage=None):
        c = self.config
        train_tf = transforms.Compose([
            transforms.Resize((c.img_size, c.img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
        ])
        val_tf = transforms.Compose([
            transforms.Resize((c.img_size, c.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
        ])

        if c.train_dir and os.path.exists(c.train_dir):
            self.train_dataset = EuroSATDataset(
                c.train_dir, train_tf, c.num_classes,
                c.selected_classes, c.max_samples_per_class, c.seed)
            self.class_names = self.train_dataset.classes
            print(f"  Training: {len(self.train_dataset)} img "
                  f"({len(self.class_names)} classi: {self.class_names})")
        else:
            print(f"  ⚠️  Train dir non trovata → dataset sintetico")
            self.train_dataset = self._synth(600)
            self.class_names = [f'C{i}' for i in range(c.num_classes)]

        if c.val_dir and os.path.exists(c.val_dir):
            self.val_dataset = EuroSATDataset(
                c.val_dir, val_tf, c.num_classes,
                c.selected_classes, c.max_samples_per_class, c.seed + 1)
            print(f"  Validation: {len(self.val_dataset)} img")
        else:
            print(f"  ⚠️  Val dir non trovata → dataset sintetico")
            self.val_dataset = self._synth(200)

    def _synth(self, n):
        class S(Dataset):
            def __init__(s, n, nc, sz, ch):
                s.data = [(torch.randn(ch, sz, sz), random.randint(0, nc-1)) for _ in range(n)]
            def __len__(s): return len(s.data)
            def __getitem__(s, i): return s.data[i]
        c = self.config
        return S(n, c.num_classes, c.img_size, c.in_channels)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.config.batch_size,
                         shuffle=True, num_workers=self.config.num_workers,
                         pin_memory=True, persistent_workers=self.config.num_workers > 0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.config.batch_size,
                         shuffle=False, num_workers=self.config.num_workers,
                         pin_memory=True, persistent_workers=self.config.num_workers > 0)

print("✓ EuroSATDataset + DataModule")

## §4 — Metrics Logger

In [ ]:
class MetricsLogger(Callback):
    """Registra metriche per ogni epoca — log su file + memoria."""

    def __init__(self, log_dir=None):
        super().__init__()
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.log_dir = log_dir
        self._csv_file = None
        self._csv_writer = None

    def on_fit_start(self, trainer, pl_module):
        if self.log_dir:
            os.makedirs(self.log_dir, exist_ok=True)
            self._csv_file = open(os.path.join(self.log_dir, 'metrics.csv'), 'w', newline='')
            self._csv_writer = csv.writer(self._csv_file)
            self._csv_writer.writerow(['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc'])

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        metrics = trainer.callback_metrics
        tl = metrics.get('train_loss_epoch', metrics.get('train_loss', torch.tensor(0))).item()
        vl = metrics.get('val_loss', torch.tensor(0)).item()
        ta = metrics.get('train_accuracy_epoch', metrics.get('train_accuracy', torch.tensor(0))).item()
        va = metrics.get('val_accuracy', torch.tensor(0)).item()

        self.train_losses.append(tl)
        self.val_losses.append(vl)
        self.train_accuracies.append(ta)
        self.val_accuracies.append(va)

        if self._csv_writer:
            self._csv_writer.writerow([trainer.current_epoch, f'{tl:.6f}', f'{vl:.6f}',
                                       f'{ta:.6f}', f'{va:.6f}'])
            self._csv_file.flush()

    def on_fit_end(self, trainer, pl_module):
        if self._csv_file:
            self._csv_file.close()

print("✓ MetricsLogger")

## §5 — Backend Manager

In [ ]:
class BackendManager:
    """Gestisce Estimator per Qiskit 2.x."""

    def __init__(self, config: QCNNConfig):
        self.config = config
        self.estimator = None
        self.backend = None
        self.session = None
        self.pass_manager = None
        self.rng = np.random.default_rng(config.seed)
        self.backend_name = "unknown"
        self.backend_type = config.backend_type
        self.available_qubits = config.num_qubits

    def initialize(self):
        print(f"Backend: {self.config.backend_type.upper()}")
        if self.config.backend_type == "statevector":
            self._setup_statevector()
        elif self.config.backend_type == "aer":
            self._setup_aer()
        elif self.config.backend_type == "aer_noise":
            self._setup_aer_noise()
        elif self.config.backend_type == "ibm":
            self._setup_ibm()
        print(f"  ✓ {self.backend_name}")
        print(f"  Qubit disponibili: {self.available_qubits}")

    def _setup_statevector(self):
        self.estimator = StatevectorEstimator()
        self.available_qubits = self.config.num_qubits
        self.backend_name = "StatevectorEstimator"

    def _init_aer_estimator(self, aer_backend):
        """Inizializza AerEstimator con fallback per diverse versioni API."""
        try:
            self.estimator = AerEstimator(aer_backend)
        except TypeError:
            self.estimator = AerEstimator()

    def _setup_aer(self):
        if not HAS_AER:
            print("  ⚠️ qiskit-aer non disponibile → fallback StatevectorEstimator")
            self._setup_statevector()
            return
        aer_backend = AerSimulator(method='automatic')
        self.backend = aer_backend
        self.available_qubits = aer_backend.num_qubits
        self._init_aer_estimator(aer_backend)
        self.backend_name = f"AerSimulator(ideal, max {self.available_qubits}q)"

    def _setup_aer_noise(self):
        try:
            from qiskit_ibm_runtime import QiskitRuntimeService
            svc_kwargs = {}
            if self.config.ibm_token:
                svc_kwargs = dict(channel=self.config.ibm_channel, token=self.config.ibm_token)
                if self.config.ibm_instance:
                    svc_kwargs['instance'] = self.config.ibm_instance
            service = QiskitRuntimeService(**svc_kwargs)
            real_backend = service.backend(self.config.noise_backend_name)
            from qiskit_aer.noise import NoiseModel
            noise_model = NoiseModel.from_backend(real_backend)
            aer_backend = AerSimulator(noise_model=noise_model, method='density_matrix')
            self.backend = aer_backend
            self.available_qubits = aer_backend.num_qubits
            self._init_aer_estimator(aer_backend)
            self.pass_manager = generate_preset_pass_manager(
                optimization_level=self.config.optimization_level, backend=aer_backend)
            self.backend_name = f"AerSimulator(noise={self.config.noise_backend_name})"
        except Exception as e:
            print(f"  ⚠️ Noise model fallback: {e}")
            self._setup_aer()

    def _setup_ibm(self):
        from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
        svc_kwargs = {}
        if self.config.ibm_token:
            svc_kwargs = dict(channel=self.config.ibm_channel, token=self.config.ibm_token)
            if self.config.ibm_instance:
                svc_kwargs['instance'] = self.config.ibm_instance
        service = QiskitRuntimeService(**svc_kwargs)
        backend = service.least_busy(min_num_qubits=self.config.ibm_min_qubits, operational=True)
        self.backend = backend
        self.available_qubits = backend.num_qubits
        self.estimator = EstimatorV2(backend=backend)
        self.pass_manager = generate_preset_pass_manager(
            optimization_level=self.config.optimization_level, backend=backend)
        self.backend_name = backend.name

    def transpile(self, circuit):
        if self.pass_manager:
            return self.pass_manager.run(circuit)
        return circuit

    def close(self):
        if self.session:
            self.session.close()
        print("Backend chiuso.")

print("✓ BackendManager")


## §6 — Circuit Builder 

Circuito 9 qubit:
1. RY(x_i) — encoding dati (patch 3×3)
2. H — superposizione
3. CNOT staircase ascendente: CX(0,1), CX(1,2), ..., CX(7,8)
4. RZ(θ_i) — parametri trainabili
5. CNOT staircase discendente: CX(7,8), CX(6,7), ..., CX(0,1)
6. Misura ⟨Z⟩ su singolo qubit

In [ ]:
class FilippiCircuitBuilder:
    """Circuito quantistico da Filippi.

    9 qubit, 9 parametri trainabili (RZ), misura singolo qubit.
    Encoding: RY(x_i), poi H, CNOT staircase, RZ(θ_i), inv CNOT staircase.
    """

    def __init__(self, config: QCNNConfig):
        self.n = config.num_qubits           # 9
        self.num_weights = config.num_weights  # 9 (RZ gates)
        self.measure_qubit = config.measure_qubit
        self.total_qubits = self.n

        # Parametri
        self.input_params = ParameterVector('x', self.n)
        self.weight_params = ParameterVector('w', self.num_weights)

        # Costruisci circuito
        self.circuit = self._build()

        # Osservabile: ⟨Z⟩ su singolo qubit → 1 output scalare
        self.observables = [
            SparsePauliOp.from_sparse_list(
                [('Z', [self.measure_qubit], 1.0)], num_qubits=self.n
            )
        ]
        self.num_observables = 1

        # Mappa parametri → indici per PUB batching veloce
        self.param_list = list(self.circuit.parameters)
        self.num_params = len(self.param_list)
        param_to_idx = {p: i for i, p in enumerate(self.param_list)}
        self.input_indices = [param_to_idx[self.input_params[i]] for i in range(self.n)]
        self.weight_indices = [param_to_idx[self.weight_params[i]] for i in range(self.num_weights)]

    def _build(self):
        """Costruisce il circuito Filippi ."""
        qc = QuantumCircuit(self.n)

        # 1. Encoding: RY(x_i) su tutti i qubit
        for i in range(self.n):
            qc.ry(self.input_params[i], i)

        # 2. Hadamard su tutti i qubit
        for i in range(self.n):
            qc.h(i)

        # 3. CNOT staircase ascendente
        for i in range(self.n - 1):
            qc.cx(i, i + 1)

        # 4. RZ trainabili su tutti i qubit
        for i in range(self.n):
            qc.rz(self.weight_params[i], i)

        # 5. CNOT staircase discendente (invertito)
        for i in range(self.n - 2, -1, -1):
            qc.cx(i, i + 1)

        return qc

    def build_param_array(self, inputs_2d, weights_1d):
        """Costruisce array parametri 2D per PUB broadcasting.

        Args:
            inputs_2d: (N, 9) — N patch, 9 input per patch
            weights_1d: (9,) — pesi condivisi
        Returns:
            params: (N, num_params) — array per estimator PUB
            N: numero di patch
        """
        N = inputs_2d.shape[0]
        params = np.zeros((N, self.num_params))
        for j, idx in enumerate(self.weight_indices):
            params[:, idx] = weights_1d[j]
        for j, idx in enumerate(self.input_indices):
            params[:, idx] = inputs_2d[:, j]
        return params, N

    def parse_output(self, evs, N):
        """Output: (N, 1) — singola expectation value per patch."""
        return evs[:N]  # (N, 1)


print("✓ FilippiCircuitBuilder")

# Test circuito
_test_cfg = QCNNConfig()
_builder = FilippiCircuitBuilder(_test_cfg)
print(f"  Circuito: {_builder.n}q, {_builder.num_weights} pesi, "
      f"{_builder.num_params} params totali")
print(f"  Osservabili: {_builder.num_observables} (Z su qubit {_builder.measure_qubit})")
print(_builder.circuit.draw(output='text', fold=120))

## §7 — Quantum Execution Engine (PUB Batching)

In [ ]:
class QuantumEngine:
    """Motore di esecuzione quantistica con PUB batching.

    Interfaccia:
    - forward(inputs, weights) → (N, 1)
    - forward_and_gradient(inputs, weights) → fwd, grad_w, grad_x
    """

    SHIFT = np.pi / 2

    def __init__(self, config: QCNNConfig, backend_manager: BackendManager):
        self.config = config
        self.backend_manager = backend_manager
        self.n = config.num_qubits

        # Builder Filippi
        self.builder = FilippiCircuitBuilder(config)
        self.circuit = self.builder.circuit
        self.observables = self.builder.observables
        self.num_weights = self.builder.num_weights
        self.num_observables = self.builder.num_observables

        # Transpile se necessario
        if backend_manager.pass_manager:
            self.circuit_exec = backend_manager.transpile(self.circuit)
        else:
            self.circuit_exec = self.circuit

        # Statistiche
        self.total_estimator_calls = 0
        self.total_pub_count = 0

    def _run_pubs(self, pubs):
        """Esegue una lista di PUB e ritorna i risultati."""
        job = self.backend_manager.estimator.run(pubs)
        results = job.result()
        self.total_estimator_calls += 1
        self.total_pub_count += len(pubs)
        return results

    def forward(self, inputs, weights):
        """Forward pass.

        Args:
            inputs: (N, 9) numpy array
            weights: (9,) numpy array
        Returns:
            (N, 1) expectation values
        """
        pv, N = self.builder.build_param_array(inputs, weights)
        # PUB: (circuit, observables, params[N, 1, P])
        pv_broad = pv[:, np.newaxis, :]  # (N, 1, P)
        results = self._run_pubs([(self.circuit_exec, self.observables, pv_broad)])
        evs = np.array(results[0].data.evs)  # (N, 1)
        return self.builder.parse_output(evs, N)

    def forward_and_gradient(self, inputs, weights, skip_input_grad=False):
        """Forward + parameter shift gradient in UNA sola chiamata estimator.

        Args:
            inputs: (N, 9)
            weights: (9,)
        Returns:
            fwd: (N, 1) — output forward
            grad_w: (W, N, 1) — jacobiano pesi
            grad_x: (9, N, 1) — jacobiano input
        """
        N = inputs.shape[0]
        nw = self.num_weights
        n = self.n
        n_obs = self.num_observables  # 1
        shift = self.SHIFT

        pubs = []

        # [0] Forward
        pv_fwd, _ = self.builder.build_param_array(inputs, weights)
        pubs.append((self.circuit_exec, self.observables, pv_fwd[:, np.newaxis, :]))

        # [1..2W] Weight shifts (parameter shift rule)
        for j in range(nw):
            w_plus = weights.copy(); w_plus[j] += shift
            w_minus = weights.copy(); w_minus[j] -= shift
            pv_p, _ = self.builder.build_param_array(inputs, w_plus)
            pv_m, _ = self.builder.build_param_array(inputs, w_minus)
            pubs.append((self.circuit_exec, self.observables, pv_p[:, np.newaxis, :]))
            pubs.append((self.circuit_exec, self.observables, pv_m[:, np.newaxis, :]))

        # [2W+1..2W+2n] Input shifts (saltati se skip_input_grad=True - ottimizzazione Wave K)
        if not skip_input_grad:
            for i in range(n):
                x_plus = inputs.copy(); x_plus[:, i] += shift
                x_minus = inputs.copy(); x_minus[:, i] -= shift
                pv_p, _ = self.builder.build_param_array(x_plus, weights)
                pv_m, _ = self.builder.build_param_array(x_minus, weights)
                pubs.append((self.circuit_exec, self.observables, pv_p[:, np.newaxis, :]))
                pubs.append((self.circuit_exec, self.observables, pv_m[:, np.newaxis, :]))

        # Esecuzione unica
        results = self._run_pubs(pubs)

        # Parse risultati
        fwd = np.array(results[0].data.evs)[:N]  # (N, 1)

        grad_w = np.zeros((nw, N, n_obs))
        for j in range(nw):
            ev_p = np.array(results[1 + 2*j].data.evs)[:N]
            ev_m = np.array(results[1 + 2*j + 1].data.evs)[:N]
            grad_w[j] = (ev_p - ev_m) / 2.0

        if skip_input_grad:
            grad_x = None  # Ottimizzazione Wave K: input shifts saltati
        else:
            grad_x = np.zeros((n, N, n_obs))
            offset = 1 + 2 * nw
            for i in range(n):
                ev_p = np.array(results[offset + 2*i].data.evs)[:N]
                ev_m = np.array(results[offset + 2*i + 1].data.evs)[:N]
                grad_x[i] = (ev_p - ev_m) / 2.0

        return fwd, grad_w, grad_x

print("✓ QuantumEngine")

## §8 — Autograd Function + Quantum Conv Layer (channel batching)

**Ottimizzazione chiave**: tutti i 6 canali vengono concatenati in un unico batch
e processati con UNA sola estimator call (invece di 6 separate).
Il circuito produce 1 scalare per patch → mantiene C_in canali in output.
Speedup: 6× per forward, 6× per backward.

In [ ]:
class QuantumConvFunction(torch.autograd.Function):
    """Autograd con parameter shift batched — output scalare per patch.

    Forward: 1 PUB (N bindings) → 1 estimator call
    Backward: 2*(9+9) = 36 PUB → 1 estimator call
    """

    @staticmethod
    def forward(ctx, input_patches, weights, engine):
        input_np = input_patches.detach().cpu().numpy()
        weights_np = weights.detach().cpu().numpy()

        outputs = engine.forward(input_np, weights_np)  # (N, 1)

        ctx.save_for_backward(input_patches, weights)
        ctx.engine = engine
        return torch.tensor(outputs, dtype=torch.float32, device=input_patches.device)

    @staticmethod
    def backward(ctx, grad_output):
        input_patches, weights = ctx.saved_tensors
        engine = ctx.engine
        device = grad_output.device

        input_np = input_patches.detach().cpu().numpy()
        weights_np = weights.detach().cpu().numpy()
        grad_out_np = grad_output.detach().cpu().numpy()  # (N, 1)

        # OTTIMIZZAZIONE WAVE K (E): salta input shifts se non servono
        # (input_patches.requires_grad=False quando il chiamante ha applicato .detach())
        skip_input = not input_patches.requires_grad

        # Forward + gradients in UNA chiamata
        _, grad_w_jac, grad_x_jac = engine.forward_and_gradient(
            input_np, weights_np, skip_input_grad=skip_input)
        # grad_w_jac: (W, N, 1), grad_x_jac: (9, N, 1) o None

        # Chain rule: pesi — somma su N e observables
        grad_weights = np.einsum('jnq,nq->j', grad_w_jac, grad_out_np)

        # Chain rule: input (skip se ottimizzazione attiva)
        if skip_input or grad_x_jac is None:
            grad_inputs = None
        else:
            grad_inputs = np.einsum('inq,nq->ni', grad_x_jac, grad_out_np)
            grad_inputs = torch.tensor(grad_inputs, dtype=torch.float32, device=device)

        return (grad_inputs,
                torch.tensor(grad_weights, dtype=torch.float32, device=device),
                None)


class QuantumConvLayer(nn.Module):
    """Layer convoluzionale quantistico — Filippi + channel batching.

    OTTIMIZZAZIONE CHIAVE: tutti i canali in UNA sola estimator call.
    Invece di 6 chiamate separate (una per canale), concateniamo tutti i
    patch di tutti i canali in un unico batch → 6× speedup.

    Pipeline:
    1. Pad input spaziale (quanv_padding)
    2. Per ogni canale: estrai patch 3×3 → 9 valori
    3. Concatena patch di TUTTI i canali: (B*C*P, 9)
    4. UNA sola quantum call → (B*C*P, 1)
    5. Reshape in (B, C, H_out, W_out)

    Pesi quantistici condivisi tra tutti i canali (depthwise, shared weights).
    """

    def __init__(self, config: QCNNConfig, backend_manager: BackendManager):
        super().__init__()
        self.config = config
        self.n = config.num_qubits           # 9
        self.kernel_size = config.kernel_size  # 3
        self.stride = config.stride            # 1
        self.padding = config.quanv_padding    # 2

        # Scaling input → range [-π, π] per encoding RY
        self.input_scale = nn.Parameter(torch.ones(self.n) * 1.0)

        # Engine quantistico
        self.engine = QuantumEngine(config, backend_manager)

        # Pesi quantistici (9 RZ gates, condivisi tra tutti i canali)
        init_w = (backend_manager.rng.random(config.num_weights) * 2 - 1) * 0.3
        self.quantum_weights = nn.Parameter(torch.tensor(init_w, dtype=torch.float32))

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W) — feature maps dai conv classici
        Returns:
            (B, C, H_out, W_out)
        """
        B, C, H, W = x.shape

        # Pad spaziale (zero padding)
        if self.padding > 0:
            x_pad = F.pad(x, [self.padding]*4, mode='constant', value=0.0)
        else:
            x_pad = x
        _, _, Hp, Wp = x_pad.shape

        # Calcola dimensioni output
        H_out = (Hp - self.kernel_size) // self.stride + 1
        W_out = (Wp - self.kernel_size) // self.stride + 1
        P = H_out * W_out  # patch per canale

        # ── Channel batching: concatena patch di tutti i canali ──
        all_patches = []
        for c in range(C):
            # Estrai singolo canale: (B, 1, Hp, Wp)
            x_c = x_pad[:, c:c+1, :, :]

            # Unfold → patch 3×3: (B, 9, P)
            patches = F.unfold(x_c, kernel_size=self.kernel_size, stride=self.stride)

            # Normalizzazione → [0, π] per encoding RY ottimale
            # BatchNorm stabilizza i valori, qui li mappiamo sulla sfera di Bloch
            p_min = patches.min(dim=2, keepdim=True).values
            p_max = patches.max(dim=2, keepdim=True).values
            p_range = (p_max - p_min).clamp(min=1e-8)
            patches_scaled = (patches - p_min) / p_range * np.pi

            # (B, P, 9)
            all_patches.append(patches_scaled.permute(0, 2, 1).contiguous())

        # Concatena: (B, C*P, 9) → flatten → (B*C*P, 9)
        all_patches = torch.cat(all_patches, dim=1)  # (B, C*P, 9)
        all_flat = all_patches.reshape(-1, self.n)    # (B*C*P, 9)

        # ── OTTIMIZZAZIONE WAVE K (E): detach input grad ──
        # Disconnette il flusso di gradiente attraverso gli input del quanv:
        # i conv classici a monte non ricevono gradient *attraverso* il quanv
        # (lo ricevono comunque dalla classification loss via skip-path).
        # Effetto: dimezza il numero di PUB del backward (36 -> 18) togliendo
        # le 18 input-shift PUB; i 9 pesi quantum trainabili rimangono invariati.
        # Trade-off documentato in sec:results-stats del Cap.3.
        all_flat = all_flat.detach()

        # ── UNA sola quantum call per tutti i canali ──
        q_out = QuantumConvFunction.apply(all_flat, self.quantum_weights, self.engine)
        # q_out: (B*C*P, 1)

        # Reshape: (B*C*P, 1) → (B, C, P) → (B, C, H_out, W_out)
        q_out = q_out.reshape(B, C, P)
        return q_out.reshape(B, C, H_out, W_out)

print("✓ QuantumConvFunction + QuantumConvLayer (Filippi, channel batching)")


## §9 — Hybrid Conv Net 

Conv1(3→6, k5, p2) → Pool(2) → Conv2(6→6, k5, p2) → Pool(2) → Quanv(6→6) → Flatten → FC

In [ ]:
class HybridConvNet(nn.Module):
    """Architettura Filippi + BatchNorm per stabilità.

    Conv1(3→6) → BN → Pool → Conv2(6→6) → BN → Pool → Quanv(6→6, 9q) → Flatten → FC
    BatchNorm aggiunto per stabilizzare le feature in input al circuito quantistico
    (essenziale con dataset piccoli).
    """

    def __init__(self, config: QCNNConfig, quantum_layer: QuantumConvLayer):
        super().__init__()
        ch = config.num_conv_channels  # 6
        ks = config.conv_kernel_size   # 5
        pad = config.conv_padding      # 2
        drop = config.dropout_rate

        # Feature extraction classica (+ BatchNorm per stabilità)
        self.conv1 = nn.Sequential(
            nn.Conv2d(config.in_channels, ch, ks, padding=pad),
            nn.BatchNorm2d(ch),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(ch, ch, ks, padding=pad),
            nn.BatchNorm2d(ch),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Dropout opzionale
        self.drop1 = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.drop2 = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.drop_q = nn.Dropout2d(drop) if drop > 0 else nn.Identity()

        # Quantum convolution
        self.quantum_conv = quantum_layer

        # Classifier
        flat = config.flatten_size
        fc1_out = max(flat // 3, 64)
        self.classifier = nn.Sequential(
            nn.Linear(flat, fc1_out),
            nn.ReLU(),
            nn.Linear(fc1_out, config.num_classes),
        )

        total_p = sum(p.numel() for p in self.parameters())
        q_p = len(quantum_layer.quantum_weights)
        print(f"HybridConvNet (Filippi + BatchNorm):")
        print(f"  Conv1: {config.in_channels}→{ch} (k{ks}) + BN")
        print(f"  Conv2: {ch}→{ch} (k{ks}) + BN")
        print(f"  Quanv: {ch}→{ch} ({config.num_qubits}q, k{config.kernel_size})")
        print(f"  Flatten: {flat} = {ch}×{config.quanv_output_size}²")
        print(f"  FC: {flat}→{fc1_out}→{config.num_classes}")
        print(f"  Totale: {total_p:,} params ({q_p} quantum)")

    def forward(self, x):
        x = self.drop1(self.conv1(x))    # (B, 6, 32, 32)
        x = self.drop2(self.conv2(x))    # (B, 6, 16, 16)
        x = self.drop_q(self.quantum_conv(x))  # (B, 6, 14, 14)
        return self.classifier(x.flatten(1))

print("✓ HybridConvNet (Filippi + BatchNorm)")


## §10 — Lightning Classifier

In [ ]:
class HybridQCNNClassifier(L.LightningModule):

    def __init__(self, model: HybridConvNet, config: QCNNConfig):
        super().__init__()
        self.model = model
        self.config = config
        self.loss_fn = nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task="multiclass", num_classes=config.num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=config.num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = logits.argmax(1)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)
        self.log('train_accuracy', self.train_acc(preds, y), on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = logits.argmax(1)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_accuracy', self.val_acc(preds, y), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.config.lr,
                               weight_decay=self.config.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.config.max_epochs)
        return [opt], [sched]

print("✓ HybridQCNNClassifier")

## §11 — Inizializzazione e test

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()
                        else "mps" if torch.backends.mps.is_available()
                        else "cpu")
print(f"Device: {device}")

backend_manager = BackendManager(config)
backend_manager.initialize()

In [ ]:
# Test circuito Filippi
builder = FilippiCircuitBuilder(config)
test_inputs = np.random.randn(3, 9)  # 3 patch, 9 input
test_weights = np.random.randn(9) * 0.1  # 9 pesi
pv, N = builder.build_param_array(test_inputs, test_weights)
print(f"Param array: {pv.shape} (N={N})")
print(f"Circuito: {builder.circuit.num_qubits}q, depth={builder.circuit.depth()}")

In [ ]:
# Test engine — forward e gradiente
engine = QuantumEngine(config, backend_manager)
fwd = engine.forward(test_inputs, test_weights)
print(f"Forward: {fwd.shape}")  # (3, 1)
print(f"  Valori: {fwd.flatten()}")

fwd2, gw, gx = engine.forward_and_gradient(test_inputs, test_weights)
print(f"Forward+Grad: fwd={fwd2.shape}, grad_w={gw.shape}, grad_x={gx.shape}")
print(f"  PUB totali: 1+{2*config.num_weights}+{2*config.num_qubits} = "
      f"{1 + 2*config.num_weights + 2*config.num_qubits} → "
      f"1 estimator call")

In [ ]:
# Test QuantumConvLayer (channel batching)
ql = QuantumConvLayer(config, backend_manager)
test_x = torch.randn(2, 6, 16, 16)  # B=2, C=6, dopo 2 pool
P = config.quanv_output_size ** 2
print(f"Patch per canale: {P} ({config.quanv_output_size}×{config.quanv_output_size})")
print(f"Totale patch per immagine: {6 * P} (6 canali × {P})")
print(f"Bindings per estimator call (B=2): {2 * 6 * P}")
t0 = time.time()
out = ql(test_x)
dt = time.time() - t0
print(f"QuantumConvLayer: {test_x.shape} → {out.shape}  ({dt:.1f}s)")
print(f"  Estimator calls: {ql.engine.total_estimator_calls} (channel batching!)")
# Atteso: (2, 6, 18, 18) con 1 sola estimator call

In [ ]:
# Test modello completo + gradient flow
ql_test = QuantumConvLayer(config, backend_manager)
model_test = HybridConvNet(config, ql_test)
model_test = model_test.to(device)

x_test = torch.randn(2, 3, 64, 64, device=device)
t0 = time.time()
logits = model_test(x_test)
dt = time.time() - t0
print(f"\nForward completo: {x_test.shape} → {logits.shape}  ({dt:.1f}s)")

loss = nn.CrossEntropyLoss()(logits, torch.tensor([0, 1], device=device))
loss.backward()
print(f"Loss: {loss.item():.4f}")

# Verifica gradienti
for name, p in model_test.named_parameters():
    if p.grad is not None:
        g = p.grad.abs().mean().item()
        if 'quantum' in name:
            print(f"  ⚡ {name}: grad={g:.6f}")
        elif g > 0:
            print(f"  📐 {name}: grad={g:.6f}")

del model_test, ql_test
gc.collect()

## §12 — Caricamento Dataset

In [ ]:
data_module = EuroSATDataModule(config)
data_module.setup()

## §13 — Loop Statistico

In [ ]:
def create_fresh_model(config, backend_manager, device, seed, verbose=True):
    """Nuovo modello con seed diverso."""
    L.seed_everything(seed, workers=True)
    backend_manager.rng = np.random.default_rng(seed)

    if not verbose:
        import io, contextlib
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            ql = QuantumConvLayer(config, backend_manager)
            mdl = HybridConvNet(config, ql)
    else:
        ql = QuantumConvLayer(config, backend_manager)
        mdl = HybridConvNet(config, ql)

    return mdl.to(device), ql


def collect_val_predictions(classifier, data_module, device):
    """Pass deterministica sul validation set DOPO trainer.fit, per
    raccogliere il vettore per-item di correttezza (lunghezza N_val).

    Questo serve a due cose:
      (a) calcolare il Wilson 95% CI per la singola run (descrittivo);
      (b) salvare predizioni per-item su CSV, così un notebook
          successivo può eseguire un Wilcoxon signed-rank paired
          cross-architecture sulle stesse immagini di validazione.

    Il data_module è già stato setup-pato da trainer.fit; il
    val_dataloader è in modalità shuffle=False, quindi l'ordine degli
    item è stabile e riproducibile (a parità di seed di costruzione del
    dataset).
    """
    classifier.eval()
    classifier = classifier.to(device)
    correct, labels = [], []
    with torch.no_grad():
        for x, y in data_module.val_dataloader():
            x, y = x.to(device), y.to(device)
            logits = classifier(x)
            preds = logits.argmax(dim=1)
            corr = (preds == y).to(torch.int64).cpu().tolist()
            correct.extend(int(c) for c in corr)
            labels.extend(int(v) for v in y.cpu().tolist())
    return correct, labels


def run_single_training(config, backend_manager, data_module, device, seed, run_idx, verbose=True):
    """Una singola run di training."""
    print(f"\n{'='*60}")
    print(f"  RUN {run_idx+1}/{config.num_stat_runs} — seed={seed}")
    print(f"{'='*60}")

    mdl, ql = create_fresh_model(config, backend_manager, device, seed, verbose)
    classifier = HybridQCNNClassifier(mdl, config)

    log_dir = os.path.join(config.output_dir, 'stat_runs', f'run_{run_idx:02d}_s{seed}')
    metrics_logger = MetricsLogger(log_dir=log_dir)

    callbacks = [
        metrics_logger,
        EarlyStopping(monitor='val_loss', patience=config.early_stop_patience,
                      mode='min', verbose=verbose),
    ]

    best_ckpt_path = None
    if run_idx == 0:
        best_ckpt = ModelCheckpoint(
            dirpath=log_dir, filename='best-{epoch}-{val_loss:.4f}',
            monitor='val_loss', mode='min', save_top_k=1)
        callbacks.append(best_ckpt)

    trainer = L.Trainer(
        max_epochs=config.max_epochs,
        callbacks=callbacks,
        logger=TensorBoardLogger(config.output_dir, name='stat_logs',
                                version=f'run_{run_idx:02d}'),
        accelerator='auto', devices=1,
        log_every_n_steps=1,
        enable_progress_bar=verbose,
        enable_checkpointing=(run_idx == 0),
    )

    t0 = time.time()
    trainer.fit(classifier, data_module)
    elapsed = time.time() - t0
    actual_epochs = trainer.current_epoch + 1

    # ── Raccolta predizioni per-item al final-epoch model ──
    # (necessaria per Wilson CI e per il Wilcoxon paired cross-architecture)
    val_correct, val_labels = collect_val_predictions(classifier, data_module, device)

    result = {
        'seed': seed, 'run_idx': run_idx, 'elapsed': elapsed,
        'actual_epochs': actual_epochs,
        'train_losses': list(metrics_logger.train_losses),
        'val_losses': list(metrics_logger.val_losses),
        'train_accuracies': list(metrics_logger.train_accuracies),
        'val_accuracies': list(metrics_logger.val_accuracies),
        'best_val_acc': max(metrics_logger.val_accuracies) if metrics_logger.val_accuracies else 0,
        'best_val_loss': min(metrics_logger.val_losses) if metrics_logger.val_losses else float('inf'),
        'final_train_acc': metrics_logger.train_accuracies[-1] if metrics_logger.train_accuracies else 0,
        'final_val_acc': metrics_logger.val_accuracies[-1] if metrics_logger.val_accuracies else 0,
        'val_correct_final': val_correct,
        'val_labels_final':  val_labels,
        'n_val': len(val_correct),
        'estimator_calls': ql.engine.total_estimator_calls,
        'pub_count': ql.engine.total_pub_count,
    }

    print(f"  ⏱  {elapsed:.0f}s ({actual_epochs} epoche)")
    print(f"  Best val_acc: {result['best_val_acc']:.4f}")
    print(f"  Final val_acc: {result['final_val_acc']:.4f} "
          f"({sum(val_correct)}/{len(val_correct)})")
    print(f"  Estimator calls: {result['estimator_calls']}")

    del classifier, trainer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return result


In [ ]:
# ═══════════════════════════════════════════
#  ESECUZIONE LOOP STATISTICO
# ═══════════════════════════════════════════
os.makedirs(config.output_dir, exist_ok=True)
results = []

for run_idx in range(config.num_stat_runs):
    seed = config.base_seed + run_idx * 111
    result = run_single_training(config, backend_manager, data_module, device,
                                 seed, run_idx, verbose=True)
    results.append(result)

print(f"\n{'='*60}")
print(f"  COMPLETATO: {len(results)} run")
print(f"{'='*60}")

## §14 — Tabella Riassuntiva

In [ ]:
def print_summary(results, config):
    print(f"\n{'─'*70}")
    print(f" {'Run':>4} │ {'Seed':>6} │ {'Epochs':>6} │ {'Best Val Acc':>12} │ "
          f"{'Best Val Loss':>13} │ {'Time':>8}")
    print(f"{'─'*70}")
    for r in results:
        print(f" {r['run_idx']+1:4d} │ {r['seed']:6d} │ {r['actual_epochs']:6d} │ "
              f"{r['best_val_acc']:12.4f} │ {r['best_val_loss']:13.4f} │ "
              f"{r['elapsed']:7.0f}s")
    print(f"{'─'*70}")
    accs = [r['best_val_acc'] for r in results]
    losses = [r['best_val_loss'] for r in results]
    print(f" Mean best val_acc:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f" Mean best val_loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

print_summary(results, config)

## §15 — Visualizzazione: tutte le curve

In [ ]:
def plot_all_curves(results, config):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Filippi — {config.num_conv_channels}ch, {config.num_qubits}q, '
                 f'{config.num_classes} classi', fontsize=13)

    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

    for i, r in enumerate(results):
        ep = range(1, len(r['train_losses'])+1)
        axes[0].plot(ep, r['train_losses'], '-', color=colors[i], alpha=0.6, label=f'Train R{i}')
        axes[0].plot(ep, r['val_losses'], '--', color=colors[i], alpha=0.8, label=f'Val R{i}')
        axes[1].plot(ep, r['train_accuracies'], '-', color=colors[i], alpha=0.6)
        axes[1].plot(ep, r['val_accuracies'], '--', color=colors[i], alpha=0.8)

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss vs Epoch'); axes[0].legend(fontsize=7)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy vs Epoch')

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'all_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

plot_all_curves(results, config)

## §16 — Media ± σ

In [ ]:
def plot_mean_bands(results, config):
    """Media e dispersione across-seed delle curve train/val per loss e accuracy.

    La banda +/- 1 sigma e' disegnata come fill_between nello stesso
    colore della linea ma con alpha ridotto: matplotlib la compone
    automaticamente come una tinta piu' chiara del colore della linea
    (richiesta esplicita per la versione del Cap.3 della tesi).

    Palette colour-blind safe (RdBu/Greens vibrant): blu per Validation,
    arancione per Training, su entrambi i pannelli.
    """
    max_ep = max(len(r['val_losses']) for r in results)

    def pad(arr, length):
        padded = np.full(length, np.nan)
        padded[:len(arr)] = arr
        return padded

    train_losses = np.array([pad(r['train_losses'], max_ep) for r in results])
    val_losses   = np.array([pad(r['val_losses'],   max_ep) for r in results])
    train_accs   = np.array([pad(r['train_accuracies'], max_ep) for r in results])
    val_accs     = np.array([pad(r['val_accuracies'],   max_ep) for r in results])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Filippi — Media \u00b1 \u03c3 ({len(results)} run)',
                 fontsize=13)
    ep = np.arange(1, max_ep + 1)

    BAND_ALPHA = 0.22
    LINE_KWARGS = {'linewidth': 1.8, 'marker': 'o', 'markersize': 3}

    panels = [
        (train_losses, 'Train Loss',     '#e08214', axes[0]),
        (val_losses,   'Validation Loss', '#2166ac', axes[0]),
        (train_accs,   'Train Accuracy', '#e08214', axes[1]),
        (val_accs,     'Validation Accuracy', '#2166ac', axes[1]),
    ]
    for data, label, color, ax in panels:
        mean = np.nanmean(data, axis=0)
        std  = np.nanstd(data, axis=0)
        ax.plot(ep, mean, color=color, label=label, **LINE_KWARGS)
        ax.fill_between(ep, mean - std, mean + std,
                        color=color, alpha=BAND_ALPHA, linewidth=0)

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss vs Epoch')
    axes[0].grid(True, linestyle='--', alpha=0.4)
    axes[0].legend(loc='upper right', fontsize=9)

    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy vs Epoch')
    axes[1].set_ylim(0.0, 1.02)
    axes[1].grid(True, linestyle='--', alpha=0.4)
    axes[1].legend(loc='lower right', fontsize=9)

    plt.tight_layout()
    fig_path = os.path.join(config.output_dir, 'mean_bands.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

plot_mean_bands(results, config)


## §17 — Miglior run + confronto Filippi

In [ ]:
def plot_best_run(results, config):
    best = max(results, key=lambda r: r['best_val_acc'])
    ep = range(1, len(best['train_losses'])+1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Best Run (seed={best["seed"]}) — '
                 f'Val Acc: {best["best_val_acc"]:.2%}', fontsize=13)

    # Loss
    axes[0].plot(ep, best['train_losses'], '-o', color='red', markersize=4, label='Train Loss')
    axes[0].plot(ep, best['val_losses'], '-o', color='green', markersize=4, label='Validation Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss vs Epoch'); axes[0].legend()

    # Accuracy
    axes[1].plot(ep, best['train_accuracies'], '-o', color='red', markersize=4, label='Train Accuracy')
    axes[1].plot(ep, best['val_accuracies'], '-o', color='green', markersize=4, label='Validation Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy vs Epoch'); axes[1].legend()

    # Evidenzia jump dopo prima epoca (come riportato nella tesi di Filippi)
    if len(best['val_accuracies']) > 0:
        first_epoch_acc = best['val_accuracies'][0]
        axes[1].annotate(f'{first_epoch_acc:.1%}',
                        xy=(1, first_epoch_acc),
                        xytext=(3, first_epoch_acc - 0.1),
                        arrowprops=dict(arrowstyle='->', color='black'),
                        fontsize=10, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'best_run.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Confronto con Filippi
    print(f"\n📊 Confronto con Filippi:")
    print(f"  Filippi: val_acc ≈ 88% dopo epoca 1, loss ≈ 0.28")
    print(f"  Nostro:  val_acc = {best['best_val_acc']:.1%}, "
          f"loss = {best['best_val_loss']:.4f}")
    if len(best['val_accuracies']) > 0:
        print(f"  Jump epoca 1: {best['val_accuracies'][0]:.1%} "
              f"(Filippi: ~87%)")

plot_best_run(results, config)

## §18 — Distribuzione risultati

In [ ]:
def plot_distributions(results, config):
    if len(results) < 3:
        print("Servono almeno 3 run per le distribuzioni")
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    accs = [r['best_val_acc'] for r in results]
    losses = [r['best_val_loss'] for r in results]

    axes[0].hist(accs, bins=min(10, len(results)), edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(np.mean(accs), color='red', linestyle='--', label=f'μ={np.mean(accs):.4f}')
    axes[0].set_xlabel('Best Val Accuracy'); axes[0].set_title('Distribuzione Accuracy')
    axes[0].legend()

    axes[1].hist(losses, bins=min(10, len(results)), edgecolor='black', alpha=0.7, color='salmon')
    axes[1].axvline(np.mean(losses), color='blue', linestyle='--', label=f'μ={np.mean(losses):.4f}')
    axes[1].set_xlabel('Best Val Loss'); axes[1].set_title('Distribuzione Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()

plot_distributions(results, config)

## §18.5 — Statistica inferenziale (Wilson + bootstrap)

Aggiunta del layer inferenziale per supportare le affermazioni di robustezza nel Cap. 3:

* **Wilson 95% CI per run**: intervallo binomiale sull'accuratezza single-run con $N_{\mathrm{val}}$ items. Per il setup di Filippi (4 classi, 100 img/classe) $N_{\mathrm{val}}\approx 400$, quindi la mezza-larghezza del CI è dell'ordine di $\pm 3$ punti percentuali a $\hat p \approx 0.9$ — informativo.
* **Bootstrap 95% percentile CI** sulla media across-run del *final* val accuracy (10\,000 resamples). Descrittivo, complementare a $\bar m \pm \sigma$.
* **Plot dei Wilson intervals** per ogni run, con la media across-run come riferimento.

In [ ]:
from scipy.stats import norm

def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval per una proporzione binomiale k/n."""
    if n <= 0:
        return 0.0, 0.0
    z = float(norm.ppf(1.0 - alpha / 2.0))
    p = k / n
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2.0 * n)) / denom
    half = z * np.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def bootstrap_ci_mean(values, n_resamples=10000, alpha=0.05, seed=0):
    """Percentile bootstrap CI sulla media."""
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    n = values.size
    boots = np.empty(n_resamples)
    for i in range(n_resamples):
        boots[i] = values[rng.integers(0, n, size=n)].mean()
    lo, hi = np.percentile(boots, [100 * alpha / 2.0, 100 * (1 - alpha / 2.0)])
    return float(values.mean()), float(values.std(ddof=1)) if n > 1 else float('nan'), float(lo), float(hi)


def summary_statistical_table(results, config):
    """Stampa una tabella per-run con Wilson 95% CI e una riga di sintesi
    across-run con bootstrap percentile CI sulla media."""
    n_val = results[0]['n_val'] if results else 0
    final_accs = np.array([r['final_val_acc'] for r in results], dtype=float)
    best_accs  = np.array([r['best_val_acc']  for r in results], dtype=float)

    print(f"\n{'\u2500' * 86}")
    print(f" Per-run final val accuracy + Wilson 95% CI (N_val = {n_val})")
    print(f"{'\u2500' * 86}")
    print(f" {'Run':>4} \u2502 {'Seed':>6} \u2502 {'k/n':>9} \u2502 "
          f"{'final acc':>10} \u2502 {'Wilson 95% CI':>22} \u2502 {'best acc':>10}")
    print(f"{'\u2500' * 86}")
    for r in results:
        k = int(round(r['final_val_acc'] * r['n_val']))
        lo, hi = wilson_ci(k, r['n_val'])
        ci_str = f"[{lo:.4f}, {hi:.4f}]"
        print(f" {r['run_idx']+1:>4d} \u2502 {r['seed']:>6d} \u2502 "
              f"{k:>4d}/{r['n_val']:<4d} \u2502 {r['final_val_acc']:>10.4f} \u2502 "
              f"{ci_str:>22} \u2502 {r['best_val_acc']:>10.4f}")
    print(f"{'\u2500' * 86}")

    m, s, lo, hi = bootstrap_ci_mean(final_accs, seed=1)
    print(f" Across-run final val acc: mean = {m:.4f}, std = {s:.4f}")
    print(f" Bootstrap 95% CI on the mean: [{lo:.4f}, {hi:.4f}]")

    m_b, s_b, lo_b, hi_b = bootstrap_ci_mean(best_accs, seed=2)
    print(f" Across-run BEST  val acc: mean = {m_b:.4f}, std = {s_b:.4f}")
    print(f" Bootstrap 95% CI on the mean: [{lo_b:.4f}, {hi_b:.4f}]")
    print(f"{'\u2500' * 86}")
    return {
        'final_mean': m, 'final_std': s, 'final_ci': (lo, hi),
        'best_mean': m_b, 'best_std': s_b, 'best_ci': (lo_b, hi_b),
        'n_val': n_val,
    }

stats_summary = summary_statistical_table(results, config)

In [ ]:
def plot_wilson_intervals(results, config):
    """Plot dei Wilson 95% CI per ogni run + media across-run con bootstrap band."""
    n_val = results[0]['n_val']
    seeds = [r['seed'] for r in results]
    point  = np.array([r['final_val_acc'] for r in results], dtype=float)
    ks = (point * n_val).round().astype(int)
    lo_arr, hi_arr = [], []
    for k in ks:
        lo, hi = wilson_ci(int(k), n_val)
        lo_arr.append(lo); hi_arr.append(hi)
    lo_arr, hi_arr = np.array(lo_arr), np.array(hi_arr)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(results))
    err_low = point - lo_arr
    err_high = hi_arr - point
    ax.errorbar(x, point, yerr=[err_low, err_high],
                fmt='o', color='#2166ac', ecolor='#2166ac',
                elinewidth=1.4, capsize=4, capthick=1.4,
                markersize=6, label='Single-run accuracy + Wilson 95% CI')

    mean = float(point.mean())
    _, _, b_lo, b_hi = bootstrap_ci_mean(point, seed=1)
    ax.axhline(mean, linestyle='--', color='#b2182b', linewidth=1.4,
               label=f'Across-run mean = {mean:.4f}')
    ax.axhspan(b_lo, b_hi, alpha=0.18, color='#b2182b',
               label=f'Bootstrap 95% CI on the mean: [{b_lo:.4f}, {b_hi:.4f}]')

    ax.set_xticks(x)
    ax.set_xticklabels([f's={s}' for s in seeds], rotation=30, fontsize=8)
    ax.set_ylabel('Validation accuracy')
    ax.set_title(f'Wilson 95% CI per run + bootstrap CI on the mean (N_val = {n_val})')
    ax.set_ylim(max(0.0, lo_arr.min() - 0.05), min(1.02, hi_arr.max() + 0.05))
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    fig_path = os.path.join(config.output_dir, 'wilson_intervals.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

plot_wilson_intervals(results, config)

## §18.7 — Confronto cross-architecture (Wilcoxon paired)

Confronto QCNN vs CCNN (CNN classica) a parità di **capacità rappresentativa** (stessi 16/32/64 canali nei blocchi convolutivi, stesso dropout 5%, stessa profondità). L'unico cambiamento è che il blocco quanvolutional finale è sostituito da un blocco `Conv2d(64, 64)` classico equivalente. Questa è l'ablation onesta della Hybrid Q-CNN.

**Nota su Filippi (tesi magistrale, Pisa AA 2024/2025)**: la tesi di Filippi citava un classical baseline Le-Net5 con canali 6/16, no dropout, due Conv layer, e su esso aveva osservato un loss 0.37 vs 0.25 per il Q-CNN. Quel confronto, però, soffre di una asimmetria di capacità — il classical baseline ha sostanzialmente meno parametri del Q-CNN. Per il confronto inferenziale di questa sezione usiamo invece un CCNN a parità di capacità (vedi `classical_cnn_multiseed_stats_v1.ipynb`).

**Nota su Sebastianelli (2021)**: il modello purely-quantum di riferimento è citato nel Cap.3 della tesi solo come riferimento bibliografico; non lo includiamo nel Wilcoxon paired perché non disponiamo dei dati grezzi seed-per-seed.

**Prerequisito**: il notebook gemello `classical_cnn_multiseed_stats_v1.ipynb` deve essere stato eseguito con lo **stesso schema di seed** di questo notebook (`base_seed + run_idx * 111`) e deve aver salvato il suo `results.json` in `Output_CCNN_v1_multiseed/`.

Le celle qui sotto:
1. **Caricano** i `results.json` di QCNN e CCNN (skip silenzioso se CCNN mancante).
2. **Calcolano** il Wilcoxon paired QCNN-vs-CCNN sui $R$ accuracy differences across-seed.
3. **Producono** la figura `cross_architecture_with_std.png` con le due curve sovrapposte e le rispettive bande $\pm 1\sigma$.

Se il `results.json` del CCNN non è ancora presente, le celle stampano un messaggio diagnostico e proseguono senza errori.

In [ ]:
# Configurazione path: edita le due righe seguenti per puntare ai
# results.json delle altre architetture (CNN classica e purely-quantum
# reference). Se un path non esiste, viene ignorato.
OTHER_ARCH_PATHS = {
    'ccnn':   './Output_CCNN_v1_multiseed/results.json',     # TO EDIT — path al notebook gemello CCNN
}

ARCH_LABELS = {
    'qcnn':   'Hybrid Q-CNN (questo notebook)',
    'ccnn':   'Classical CNN (ablation)',
}

ARCH_COLORS = {
    'qcnn':   '#2166ac',
    'ccnn':   '#b2182b',
}

def _summarize_arch(arch_results, label):
    """Stampa una riga di diagnostica per una architettura caricata."""
    R = len(arch_results)
    nv = arch_results[0]['n_val']
    final = np.array([r['final_val_acc'] for r in arch_results])
    print(f'  \u2713 {label:>10s}: R={R} run, N_val={nv}, '
          f'final_acc mean={final.mean():.4f}, std={final.std(ddof=1):.4f}')

# Sempre presente: i risultati di questo notebook (architettura 'qcnn')
arch_results = {'qcnn': results}
print('Architetture caricate:')
_summarize_arch(arch_results['qcnn'], 'qcnn')

import json as _json2
for arch, path in OTHER_ARCH_PATHS.items():
    if os.path.exists(path):
        try:
            with open(path) as f:
                payload = _json2.load(f)
            other_results = payload['results']
            # Verifica che abbia il formato esteso (val_correct_final presente)
            if 'val_correct_final' not in other_results[0]:
                print(f'  \u26a0\ufe0f  {arch}: results.json privo di val_correct_final, '
                      f'sar\u00e0 usato solo per il plot, non per Wilson per-item.')
            arch_results[arch] = other_results
            _summarize_arch(other_results, arch)
        except Exception as e:
            print(f'  \u2717 {arch}: errore nel caricamento di {path}: {e}')
    else:
        print(f'  \u2014 {arch}: path non trovato ({path}), salto.')

print(f'\nTotale architetture disponibili per il confronto: {len(arch_results)}')

In [ ]:
# Wilcoxon signed-rank paired test su tutte le coppie di architetture
# disponibili. Il pairing \u00e8 sul run_idx: per ogni i, si confronta
# acc_A[i] con acc_B[i] (entrambi prodotti dallo stesso seed = base_seed + i*111).

from scipy.stats import wilcoxon
import itertools as _it

wilcoxon_results = {}

if len(arch_results) < 2:
    print('Servono almeno 2 architetture per il Wilcoxon paired. '
          'Esegui i notebook gemelli e aggiorna OTHER_ARCH_PATHS.')
else:
    archs = list(arch_results.keys())
    print(f'\n{"\u2500"*100}')
    print(f' Wilcoxon signed-rank paired (final val accuracy across seeds)')
    print(f'{"\u2500"*100}')
    print(f' {"A vs B":>26s} | {"R":>3s} | {"mean(A)":>8s} | {"mean(B)":>8s} | '
          f'{"mean diff":>10s} | {"W stat":>8s} | {"p (two-sided)":>14s}')
    print(f'{"\u2500"*100}')

    for A, B in _it.combinations(archs, 2):
        accA = np.array([r['final_val_acc'] for r in arch_results[A]], dtype=float)
        accB = np.array([r['final_val_acc'] for r in arch_results[B]], dtype=float)
        R = min(len(accA), len(accB))
        if R < 2:
            print(f'  {A} vs {B}: R<2 dopo allineamento, salto.')
            continue
        accA, accB = accA[:R], accB[:R]
        diffs = accA - accB
        if np.all(diffs == 0):
            stat, pval = 0.0, 1.0
        else:
            try:
                res = wilcoxon(accA, accB, alternative='two-sided',
                               zero_method='wilcox', method='exact')
            except TypeError:
                res = wilcoxon(accA, accB, alternative='two-sided',
                               zero_method='wilcox')
            stat, pval = float(res.statistic), float(res.pvalue)
        label = f'{A} vs {B}'
        print(f' {label:>26s} | {R:>3d} | {accA.mean():>8.4f} | {accB.mean():>8.4f} | '
              f'{diffs.mean():>+10.4f} | {stat:>8.1f} | {pval:>14.4g}')
        wilcoxon_results[f'{A}_vs_{B}'] = {
            'n_pairs': int(R), 'mean_A': float(accA.mean()),
            'mean_B': float(accB.mean()), 'mean_diff': float(diffs.mean()),
            'W_statistic': stat, 'p_value_two_sided': pval,
            'differences_per_seed': diffs.tolist(),
        }
    print(f'{"\u2500"*100}')
    print('\nNota: con R=10 il p-value pi\u00f9 piccolo raggiungibile (esatto,\n'
          'two-sided) \u00e8 2/2^10 \u2248 0.002, ottenuto quando tutti i 10\n'
          'sign di \u0394_i concordano. Un p~0.5 NON dimostra equivalenza:\n'
          'segnala solo che il sign della differenza non \u00e8 consistente.')

In [ ]:
def plot_cross_architecture_bands(arch_results, config):
    """Plot delle tre (o N) curve di validation accuracy vs epoch, con
    banda \u00b11\u03c3 across-seed disegnata come tinta pi\u00f9 chiara del colore
    della linea. La palette \u00e8 colour-blind safe (RdBu + Greens).
    """
    if len(arch_results) < 2:
        print('Plot cross-arch saltato: serve almeno 2 architetture caricate.')
        return None

    # Lunghezza comune (padding NaN se differiscono)
    max_ep = max(len(r['val_accuracies']) for arch in arch_results.values() for r in arch)

    def pad(arr, length):
        padded = np.full(length, np.nan); padded[:len(arr)] = arr; return padded

    fig, ax = plt.subplots(figsize=(9, 5.2))
    ep = np.arange(1, max_ep + 1)
    BAND_ALPHA = 0.22
    LINE_KW = {'linewidth': 1.8, 'marker': 'o', 'markersize': 3}

    for arch, runs in arch_results.items():
        color = ARCH_COLORS.get(arch, None)
        label = ARCH_LABELS.get(arch, arch)
        val_accs = np.array([pad(r['val_accuracies'], max_ep) for r in runs])
        mean = np.nanmean(val_accs, axis=0)
        std = np.nanstd(val_accs, axis=0)
        ax.plot(ep, mean, color=color, label=f'{label} (R={len(runs)})', **LINE_KW)
        ax.fill_between(ep, mean - std, mean + std, color=color,
                        alpha=BAND_ALPHA, linewidth=0)

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation accuracy')
    ax.set_title('Validation accuracy vs epoch \u2014 cross-architecture comparison')
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    fig_path = os.path.join(config.output_dir, 'cross_architecture_with_std.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
    return fig_path

plot_cross_architecture_bands(arch_results, config)

## §19 — Salvataggio risultati

In [ ]:
# Salva risultati completi
import json as _json
import csv as _csv

save_path = os.path.join(config.output_dir, 'results.json')
with open(save_path, 'w') as f:
    _json.dump({
        'architecture': 'hybrid_qcnn_v1',
        'config': {
            'num_qubits': config.num_qubits,
            'kernel_size': config.kernel_size,
            'num_conv_channels': config.num_conv_channels,
            'num_classes': config.num_classes,
            'max_epochs': config.max_epochs,
            'batch_size': config.batch_size,
            'lr': config.lr,
            'max_samples_per_class': config.max_samples_per_class,
            'backend_type': config.backend_type,
            'quanv_padding': config.quanv_padding,
            'measure_qubit': config.measure_qubit,
            'num_stat_runs': config.num_stat_runs,
            'base_seed': config.base_seed,
        },
        'results': results,
        'stats_summary': stats_summary if 'stats_summary' in dir() else {},
        'wilcoxon_results': wilcoxon_results if 'wilcoxon_results' in dir() else {},
    }, f, indent=2)
print(f"Risultati salvati in: {save_path}")

# Per ogni run, scrivi anche un predictions CSV con il vettore per-item
# di correttezza, in modo che un notebook successivo possa fare il
# Wilcoxon paired cross-architecture caricando solo i CSV invece dell\'intero JSON.
pred_dir = os.path.join(config.output_dir, 'predictions')
os.makedirs(pred_dir, exist_ok=True)
for r in results:
    csv_path = os.path.join(pred_dir, f"predictions_run{r['run_idx']:02d}_s{r['seed']}.csv")
    with open(csv_path, 'w', newline='') as f:
        w = _csv.writer(f)
        w.writerow(['item_idx', 'label', 'correct'])
        for i, (lab, corr) in enumerate(zip(r['val_labels_final'], r['val_correct_final'])):
            w.writerow([i, lab, corr])
print(f"Predictions per-run salvate in: {pred_dir}/")


## §19.5 — Rigenerazione plot da JSON salvati (post-hoc)

Sezione opzionale per **rigenerare i plot da uno o più `results.json` già salvati**, senza dover rieseguire il training. Utile per:

* Cambiare lo stile dei plot (palette, alpha della banda, layout) e rivederli sui dati già prodotti.
* Generare il plot cross-architecture combinando run fatte in tempi diversi o su macchine diverse.
* Rifare la statistica inferenziale (Wilson, bootstrap, Wilcoxon) su un dataset già acquisito.
* Riprodurre i numeri di un run passato senza spendere tempo di calcolo.

Le celle sono **autosufficienti**: ricostruiscono il `config` come `SimpleNamespace` dai dati salvati nel JSON, e chiamano le stesse funzioni di plot definite nelle sezioni precedenti. Funzionano sia su `results.json` prodotti da questo notebook, sia su quelli prodotti dai notebook gemelli (CNN classica, purely-quantum), purché il formato sia quello esteso del Cap.3 (presente da §19 in poi).

In [ ]:
# ─── Utilities per il replay ─────────────────────────────────────
from types import SimpleNamespace
import json as _replay_json


def load_results_from_json(json_path):
    """Carica un results.json prodotto dalla §19 e ritorna (results, config).

    Il config restituito \u00e8 un SimpleNamespace con tutti i campi salvati nel
    JSON; viene aggiunto output_dir = directory che contiene il JSON, cos\u00ec
    le figure rigenerate vengono scritte accanto al file di origine.
    Vengono settati default sensati per i campi del titolo dei plot
    (num_conv_channels, num_qubits, num_classes) se assenti nel JSON.
    """
    if not os.path.exists(json_path):
        raise FileNotFoundError(f'JSON non trovato: {json_path}')
    with open(json_path) as f:
        payload = _replay_json.load(f)

    rs = payload.get('results', [])
    if not rs:
        raise ValueError(f'Nessun \"results\" nel JSON: {json_path}')

    cfg_dict = dict(payload.get('config', {}))
    cfg_dict.setdefault('output_dir', os.path.dirname(os.path.abspath(json_path)))
    cfg_dict.setdefault('num_conv_channels', 6)
    cfg_dict.setdefault('num_qubits', 9)
    cfg_dict.setdefault('num_classes', 4)

    cfg = SimpleNamespace(**cfg_dict)
    print(f'Caricati {len(rs)} run da {json_path}')
    print(f'  output_dir per le figure rigenerate: {cfg.output_dir}')
    return rs, cfg


def replay_single_arch_plots(json_path, run_summary=True, run_curves=False):
    """Rigenera tutti i plot single-arch a partire da un results.json salvato.
    Ritorna la tupla (results, config, stats_summary).

    run_curves=True attiva anche plot_all_curves (pesante se R > 10).
    """
    rs_r, cfg_r = load_results_from_json(json_path)

    if run_summary:
        print('\n--- print_summary ---')
        print_summary(rs_r, cfg_r)
    if run_curves:
        print('\n--- plot_all_curves ---')
        plot_all_curves(rs_r, cfg_r)
    print('\n--- plot_mean_bands ---')
    plot_mean_bands(rs_r, cfg_r)
    print('\n--- plot_best_run ---')
    plot_best_run(rs_r, cfg_r)
    print('\n--- plot_distributions ---')
    plot_distributions(rs_r, cfg_r)
    print('\n--- summary_statistical_table ---')
    stats_r = summary_statistical_table(rs_r, cfg_r)
    print('\n--- plot_wilson_intervals ---')
    plot_wilson_intervals(rs_r, cfg_r)
    return rs_r, cfg_r, stats_r

print('Utilities di replay caricate: load_results_from_json, replay_single_arch_plots')

In [ ]:
# ─── Esempio: replay single-architecture ─────────────────────────
# Decommenta e adatta il path al tuo results.json salvato.

# results_replay, config_replay, stats_replay = replay_single_arch_plots(
#     './Output_QCNN_v1_multiseed/results.json'
# )

In [ ]:
# ─── Replay cross-architecture (Wilcoxon paired + plot combinato) ─

def replay_cross_arch_plots(json_paths, output_dir=None, verbose=True):
    """Carica pi\u00f9 results.json e produce:
      (a) tabella Wilcoxon paired per ciascuna coppia di architetture,
      (b) figura cross_architecture_with_std.png (validation accuracy +
          banda \u00b11\u03c3 per ciascuna architettura),
      (c) la stessa wilcoxon_results del run live, ritornata in uscita.

    Parametri:
      json_paths   dict {arch_name: path/results.json}
                   arch_name \u2208 chiavi di ARCH_LABELS ('qcnn', 'ccnn', 'pure_q')
      output_dir   directory dove scrivere la figura cross-arch.
                   Se None, usa la dir del primo JSON trovato.
    """
    arch_results_r = {}
    if verbose:
        print('Caricamento architetture per il replay:')
    for arch, path in json_paths.items():
        if not os.path.exists(path):
            if verbose:
                print(f'  \u2014 {arch}: path non trovato ({path}), salto.')
            continue
        try:
            with open(path) as f:
                payload = _replay_json.load(f)
            arch_results_r[arch] = payload['results']
            if verbose:
                R = len(arch_results_r[arch])
                nv = arch_results_r[arch][0].get('n_val', '?')
                final = np.array([r['final_val_acc'] for r in arch_results_r[arch]])
                print(f'  \u2713 {arch:>10s}: R={R}, N_val={nv}, '
                      f'mean={final.mean():.4f}, std={final.std(ddof=1):.4f}')
        except Exception as e:
            print(f'  \u2717 {arch}: errore nel caricamento: {e}')

    if len(arch_results_r) < 2:
        print('\nServono almeno 2 architetture per il replay cross-arch. '
              'Esegui gli altri notebook e/o controlla i path.')
        return arch_results_r, {}

    # ── Wilcoxon paired ──
    from scipy.stats import wilcoxon as _wilc
    import itertools as _it_replay
    wlx = {}
    print(f'\n{"\u2500" * 100}')
    print(f' Wilcoxon signed-rank paired (replay)')
    print(f'{"\u2500" * 100}')
    print(f' {"A vs B":>26s} | {"R":>3s} | {"mean(A)":>8s} | {"mean(B)":>8s} | '
          f'{"mean diff":>10s} | {"W stat":>8s} | {"p (two-sided)":>14s}')
    print(f'{"\u2500" * 100}')
    for A, B in _it_replay.combinations(list(arch_results_r.keys()), 2):
        accA = np.array([r['final_val_acc'] for r in arch_results_r[A]], dtype=float)
        accB = np.array([r['final_val_acc'] for r in arch_results_r[B]], dtype=float)
        R = min(len(accA), len(accB))
        if R < 2:
            continue
        accA, accB = accA[:R], accB[:R]
        diffs = accA - accB
        if np.all(diffs == 0):
            stat, pval = 0.0, 1.0
        else:
            try:
                res = _wilc(accA, accB, alternative='two-sided',
                            zero_method='wilcox', method='exact')
            except TypeError:
                res = _wilc(accA, accB, alternative='two-sided',
                            zero_method='wilcox')
            stat, pval = float(res.statistic), float(res.pvalue)
        label = f'{A} vs {B}'
        print(f' {label:>26s} | {R:>3d} | {accA.mean():>8.4f} | {accB.mean():>8.4f} | '
              f'{diffs.mean():>+10.4f} | {stat:>8.1f} | {pval:>14.4g}')
        wlx[f'{A}_vs_{B}'] = {
            'n_pairs': int(R), 'mean_A': float(accA.mean()),
            'mean_B': float(accB.mean()), 'mean_diff': float(diffs.mean()),
            'W_statistic': stat, 'p_value_two_sided': pval,
            'differences_per_seed': diffs.tolist(),
        }
    print(f'{"\u2500" * 100}')

    # ── Plot cross-architecture ──
    if output_dir is None:
        # Usa la directory del primo JSON come fallback
        first_path = next(p for p in json_paths.values() if os.path.exists(p))
        output_dir = os.path.dirname(os.path.abspath(first_path))
    os.makedirs(output_dir, exist_ok=True)
    cfg_replay = SimpleNamespace(
        output_dir=output_dir,
        num_qubits=9, num_conv_channels=6, num_classes=4,
    )
    plot_cross_architecture_bands(arch_results_r, cfg_replay)

    return arch_results_r, wlx


# ─── Esempio d'uso ───────────────────────────────────────────────
# Decommenta e adatta i path ai tuoi results.json:

# arch_replay, wilcoxon_replay = replay_cross_arch_plots(
#     json_paths={
#         'qcnn':   './Output_QCNN_v1_multiseed/results.json',
#         'ccnn':   './Output_CCNN_v1_multiseed/results.json',
#     },
#     output_dir='./Output_cross_arch_replay',
# )

## §20 — Cleanup

In [ ]:
backend_manager.close()